# FUGN · Synergistic underwater image enhancement

**An executable methodology implementation and a transparent 10-epoch UIEB pilot.**

Based on Chengpei Xu et al., *Fusion-based graph neural networks for synergistic underwater image enhancement*, Information Fusion 117 (2025), 102857. [Paper DOI](https://doi.org/10.1016/j.inffus.2024.102857) · [Authors' implementation](https://github.com/whZ62/FUGN) · [UIEB dataset](https://li-chongyi.github.io/proj_benchmark.html).

Run the cells in order. They download real underwater photographs, train the full model, evaluate held-out photographs, and export a self-contained HTML report with every comparison. No generated or synthetically degraded photographs are substituted for UIEB.

Install the dependencies with `pip install -r requirements.txt` before opening the notebook. The official archives are RAR5; on Debian/Ubuntu install `libarchive13` (`sudo apt-get install libarchive13`) to enable streaming RAR/ZIP extraction. GPU execution is strongly recommended. The Modal launcher pulls this notebook from its public GitHub commit and stores artifacts on `/mnt/underwater`.

The pilot learning rate is **2e-4**, as in the authors' code. The `paper` preset uses the PDF's **1e-6** and **100 epochs**. Neither preset promises the paper's benchmark: its exact split and several graph settings are unpublished, and its results average ten runs.

In [ ]:
# Configuration and dataset preparation use only the standard library, Pillow,
# and gdown. Modal can execute this cell on CPU before allocating a GPU.
import os
import sys
import json
import math
import random
import hashlib
import zipfile
from pathlib import Path
from PIL import Image, ImageOps

DATASET_PAGE = "https://li-chongyi.github.io/proj_benchmark.html"
DRIVE_IDS = {
    "raw": "12W_kkblc2Vryb9zHQ6BfGQ_NKUfXYk13",
    "reference": "1cA-8CzajnVEL4feBRKdBxjEe6hwql6Z7",
    "challenging": "1Ew_r83nXzVk0hlkfuomWqsAIxuq6kaN4",
}
EXPECTED_COUNTS = {"raw": 890, "reference": 890, "challenging": 60}
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


def make_config():
    preset = os.environ.get("FUGN_PRESET", "pilot")
    if preset not in {"pilot", "paper"}:
        raise ValueError("FUGN_PRESET must be pilot or paper")
    cfg = {
        "preset": preset, "epochs": 10 if preset == "pilot" else 100,
        "learning_rate": 2e-4 if preset == "pilot" else 1e-6,
        "batch_size": 2, "seed": 42, "image_size": 256, "block_size": 16,
        "encoder_channels": 64, "node_features": 1024, "gcn_hidden": 1500,
        "neighbors": 5, "hop": 255, "alpha": 0.5, "beta": 0.5,
        "gaussian_sigma": 1.0, "gaussian_size": 7,
        "rbf_bandwidth": "per-image positive median squared distance",
        "dropout": 0.4, "charbonnier_epsilon": 1e-3,
        "grad_clip": 0.1, "num_workers": 2, "amp": True,
        "data_dir": os.environ.get("FUGN_DATA_DIR", "data/UIEB"),
        "output_dir": os.environ.get("FUGN_OUTPUT_DIR", "results/pilot"),
        "resume": os.environ.get("FUGN_RESUME", "1") == "1",
        "commit": os.environ.get("FUGN_COMMIT", "local"),
    }
    if "FUGN_EPOCHS" in os.environ:
        cfg["epochs"] = int(os.environ["FUGN_EPOCHS"])
    if cfg["epochs"] < 1:
        raise ValueError("epochs must be positive")
    return cfg


def image_files(directory):
    return sorted(p for p in Path(directory).rglob("*") if p.suffix.lower() in IMAGE_SUFFIXES)


def download_subset(root, subset):
    import libarchive
    destination = root / subset
    marker = destination / ".complete.json"
    if marker.exists() and len(image_files(destination)) == EXPECTED_COUNTS[subset]:
        return
    destination.mkdir(parents=True, exist_ok=True)
    archive = root / f"{subset}.archive"
    # Reuse downloads from the first version, including official RAR payloads
    # whose old temporary filename happened to end in .zip.
    for legacy in (root / f"{subset}.zip", root / f"{subset}.download.zip"):
        if not archive.exists() and legacy.exists():
            legacy.replace(archive)
    if not archive.exists():
        import gdown
        part = root / f"{subset}.download"
        print(f"Downloading official UIEB {subset} images ...", flush=True)
        downloaded = gdown.download(id=DRIVE_IDS[subset], output=str(part), quiet=False, resume=True)
        if not downloaded:
            raise RuntimeError(
                f"Official Google Drive download failed for {subset}. "
                f"Download the RAR/ZIP from {DATASET_PAGE} and place it at {archive}."
            )
        part.replace(archive)
    # libarchive streams both RAR5 (the actual official files) and ZIP. Write
    # only regular image payloads to flattened basenames; never extract paths.
    names = set()
    with libarchive.file_reader(str(archive)) as entries:
        for entry in entries:
            path = Path(entry.pathname)
            if (not entry.isfile or path.suffix.lower() not in IMAGE_SUFFIXES
                    or "__MACOSX" in path.parts or path.name.startswith(".")):
                continue
            if path.name in names:
                raise ValueError(f"Duplicate image basename in archive: {path.name}")
            names.add(path.name)
            target = destination / path.name
            temporary = target.with_suffix(target.suffix + ".tmp")
            with temporary.open("wb") as output:
                for chunk in entry.get_blocks():
                    output.write(chunk)
            temporary.replace(target)
    if len(names) != EXPECTED_COUNTS[subset]:
        raise ValueError(f"Unexpected image count in {archive}: {len(names)}")
    marker.write_text(json.dumps({"source": DATASET_PAGE, "drive_id": DRIVE_IDS[subset],
                                  "count": len(names)}, indent=2))
    print(f"Validated {len(names)} {subset} files", flush=True)


def prepare_data(cfg):
    root = Path(cfg["data_dir"]).resolve()
    root.mkdir(parents=True, exist_ok=True)
    for subset in EXPECTED_COUNTS:
        download_subset(root, subset)
    maps = {}
    for subset in EXPECTED_COUNTS:
        files = image_files(root / subset)
        mapping = {p.stem: p for p in files}
        if len(mapping) != len(files) or len(files) != EXPECTED_COUNTS[subset]:
            raise ValueError(f"Duplicate stems or unexpected count in {subset}")
        maps[subset] = mapping
    if maps["raw"].keys() != maps["reference"].keys():
        raise ValueError("Raw/reference filenames are not paired one-to-one")
    for name, raw_path in maps["raw"].items():
        with Image.open(raw_path) as raw, Image.open(maps["reference"][name]) as ref:
            if raw.size != ref.size:
                raise ValueError(f"Paired image dimensions differ: {name}")
            raw.verify()
            ref.verify()
    for path in maps["challenging"].values():
        with Image.open(path) as image:
            image.verify()
    names = sorted(maps["raw"])
    random.Random(cfg["seed"]).shuffle(names)
    manifest = {
        "seed": cfg["seed"], "source": DATASET_PAGE,
        "train": sorted(names[170:]), "val": sorted(names[90:170]),
        "test": sorted(names[:90]), "challenging": sorted(maps["challenging"]),
        "files": {subset: {name: {"filename": p.name, "bytes": p.stat().st_size}
                           for name, p in mapping.items()} for subset, mapping in maps.items()},
    }
    manifest["fingerprint"] = hashlib.sha256(
        json.dumps(manifest, sort_keys=True).encode()).hexdigest()
    manifest_path = root / f"split_seed_{cfg['seed']}.json"
    if manifest_path.exists() and json.loads(manifest_path.read_text()) != manifest:
        raise ValueError("Existing split manifest differs; inspect changed data before continuing")
    manifest_path.write_text(json.dumps(manifest, indent=2))
    for first, second in [("train", "val"), ("train", "test"), ("val", "test")]:
        assert not set(manifest[first]) & set(manifest[second])
    result = {key: [(maps["raw"][name], maps["reference"][name], name)
                    for name in manifest[key]] for key in ("train", "val", "test")}
    result["challenging"] = [(maps["challenging"][name], None, name)
                              for name in manifest["challenging"]]
    result["manifest"] = manifest
    print({key: len(result[key]) for key in ("train", "val", "test", "challenging")})
    return result

## 1 · Reproducible real-image data

UIEB has **890 paired photographs** and **60 unpaired challenging photographs**. The PDF's description of 890 training images and 60 test images cannot support full-reference test scores: the challenging 60 have no references. We instead hold out **90 paired tests**, **80 paired validation images**, and **720 paired training images** with seed 42. References are selected enhancement targets, not physical ground-truth clean scenes.

Training uses paired random 256×256 crops. Images smaller than the crop are proportionally enlarged first. Validation and test images are deterministically resized to 256×256 (aspect ratio can change). All metrics and comparisons use those same tensors; no full-resolution performance is claimed.

The authors restrict redistribution of UIEB. Downloads, executed notebooks, checkpoints and reports are local/private artifacts and are excluded from the public source repository.

In [ ]:
cfg = make_config()
splits = prepare_data(cfg)
output_dir = Path(cfg["output_dir"]).resolve()
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "config.json").write_text(json.dumps(cfg, indent=2))
(output_dir / "split_manifest.json").write_text(json.dumps(splits["manifest"], indent=2))
print(json.dumps(cfg, indent=2))

In [ ]:
import time
import platform
import copy
import base64
import io
import html
import subprocess
import importlib.metadata
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import piq
from IPython.display import display, Markdown

plt.rcParams.update({"figure.dpi": 105, "savefig.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


def read_rgb(path):
    with Image.open(path) as image:
        return ImageOps.exif_transpose(image).convert("RGB")


def to_tensor(image):
    return torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1) / 255.0


class PairedImages(Dataset):
    def __init__(self, records, cfg, training=False):
        self.records, self.cfg, self.training = records, cfg, training
        self.epoch = 0

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        raw_path, ref_path, name = self.records[index]
        raw = read_rgb(raw_path)
        ref = read_rgb(ref_path) if ref_path else None
        if ref is not None and raw.size != ref.size:
            raise ValueError(f"Mismatched pair: {name}")
        size = self.cfg["image_size"]
        if self.training:
            w, h = raw.size
            scale = max(1.0, size / w, size / h)
            if scale > 1:
                dimensions = (math.ceil(w * scale), math.ceil(h * scale))
                raw = raw.resize(dimensions, Image.Resampling.BICUBIC)
                ref = ref.resize(dimensions, Image.Resampling.BICUBIC)
            # Per-image/epoch RNG makes crops independent of worker scheduling
            # and allows an epoch-boundary checkpoint to resume reproducibly.
            rng = random.Random(self.cfg["seed"] + self.epoch * len(self) + index)
            left = rng.randrange(raw.width - size + 1)
            top = rng.randrange(raw.height - size + 1)
            box = (left, top, left + size, top + size)
            raw, ref = raw.crop(box), ref.crop(box)
        else:
            raw = raw.resize((size, size), Image.Resampling.BICUBIC)
            if ref is not None:
                ref = ref.resize((size, size), Image.Resampling.BICUBIC)
        raw_tensor = to_tensor(raw)
        ref_tensor = to_tensor(ref) if ref is not None else torch.empty(0)
        return raw_tensor, ref_tensor, name


def make_loader(dataset, batch_size, shuffle=False, seed=42):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=dataset.cfg["num_workers"],
                      pin_memory=torch.cuda.is_available(),
                      generator=torch.Generator().manual_seed(seed),
                      persistent_workers=False)

## 2 · Operator-based patch graph — Equations 1–6

Each 256×256 photograph is partitioned into a row-major 16×16 grid of **256 nodes**. Floating-point RGB luminance is used for the operator descriptors, avoiding the unsigned-byte gradient clipping and BGR/RGB confusion in the public code. Sobel magnitudes describe edges; Gaussian-filtered patches describe smooth intensity structure.

For each descriptor family, the similarity is an RBF of squared Euclidean distance. The two similarities are averaged (α=0.5). We keep the five most similar **other** nodes per row, then add self-loops only during GCN normalization. Row-wise top-k can produce directed adjacency; we retain that behavior and use the PDF's row-degree normalization, without silently symmetrizing it.

The PDF omits numeric k, h, Gaussian and RBF bandwidths. Explicit implementation defaults are **k=5**, **h=255** (all nodes eligible), a **7×7 Gaussian with σ=1**, and a separate per-image positive median-distance bandwidth for each descriptor family. In the RBF, `2σ²` equals the median positive squared distance (floor 1e-6). These are documented engineering choices, not recovered author hyperparameters.

Graphs are computed from the exact input batch **after cropping**, never from references or a separately shuffled loader. The node batch dimension stays separate throughout the GCN.

In [ ]:
@torch.no_grad()
def build_graph(images, cfg, return_similarity=False):
    if images.ndim != 4 or images.shape[1] != 3:
        raise ValueError("Expected images shaped [batch, 3, height, width]")
    block = cfg["block_size"]
    batch, _, height, width = images.shape
    if height % block or width % block:
        raise ValueError("Image dimensions must be divisible by block size")
    x = images.float()
    gray = (x * x.new_tensor([0.299, 0.587, 0.114])[None, :, None, None]).sum(1, keepdim=True)
    patches = F.unfold(gray, kernel_size=block, stride=block).transpose(1, 2)
    nodes = patches.shape[1]
    patch_images = patches.reshape(batch * nodes, 1, block, block)
    sobel_x = x.new_tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])[None, None]
    sobel_y = sobel_x.transpose(-1, -2)
    padded = F.pad(patch_images, (1, 1, 1, 1), mode="reflect")
    gx, gy = F.conv2d(padded, sobel_x), F.conv2d(padded, sobel_y)
    edges = torch.sqrt(gx.square() + gy.square()).reshape(batch, nodes, -1)
    radius = cfg["gaussian_size"] // 2
    axis = torch.arange(-radius, radius + 1, device=x.device, dtype=x.dtype)
    kernel_1d = torch.exp(-axis.square() / (2 * cfg["gaussian_sigma"] ** 2))
    kernel_1d /= kernel_1d.sum()
    kernel = torch.outer(kernel_1d, kernel_1d)[None, None]
    smooth = F.conv2d(F.pad(patch_images, (radius,) * 4, mode="reflect"), kernel)
    smooth = smooth.reshape(batch, nodes, -1)
    ids = torch.arange(nodes, device=x.device)
    off_diagonal = ids[:, None] != ids[None, :]
    eligible = off_diagonal & ((ids[:, None] - ids[None, :]).abs() <= cfg["hop"])
    if not 0 < cfg["neighbors"] <= int(eligible.sum(-1).min()):
        raise ValueError("Every node must have at least k eligible non-self neighbors")

    def rbf(descriptors):
        distances = torch.cdist(descriptors, descriptors).square().clamp_min_(0)
        # Median bandwidth excludes self-distances and zero-distance duplicates.
        candidates = distances[:, off_diagonal]
        medians = []
        for values in candidates:
            positive = values[values > 1e-12]
            medians.append(positive.median() if positive.numel() else values.new_tensor(1.0))
        denominators = torch.stack(medians).clamp_min(1e-6)[:, None, None]
        return torch.exp(-distances / denominators)

    similarity = cfg["alpha"] * rbf(edges) + (1 - cfg["alpha"]) * rbf(smooth)
    scores = similarity.masked_fill(~eligible[None], -torch.inf)
    # Stable sort breaks exact ties by row-major node index, including flat images.
    neighbors = torch.argsort(scores, dim=-1, descending=True, stable=True)[..., :cfg["neighbors"]]
    adjacency = torch.zeros_like(similarity).scatter_(-1, neighbors, 1.0)
    if return_similarity:
        return adjacency, similarity
    return adjacency


def normalize_graph(adjacency):
    nodes = adjacency.shape[-1]
    with_loops = adjacency + torch.eye(nodes, device=adjacency.device)[None]
    degree = with_loops.sum(-1).clamp_min(1)
    inverse_sqrt = degree.rsqrt()
    return inverse_sqrt[:, :, None] * with_loops * inverse_sqrt[:, None, :]


def features_to_nodes(features):
    return features.flatten(2).transpose(1, 2)


def nodes_to_features(nodes, height, width):
    return nodes.transpose(1, 2).reshape(nodes.shape[0], nodes.shape[-1], height, width)

## 3 · CNN + GCN fusion — Equations 7–9

| Stage | Tensor shape (per image) |
|---|---|
| RGB input | 3 × 256 × 256 |
| Two convolutions + author residual block | 64 × 256 × 256 |
| Kernel-16 / stride-16 convolution | 1024 × 16 × 16 |
| Row-major graph features | 256 × 1024 |
| GCN → ReLU → dropout → GCN | 256 × 1500 → 256 × 1024 |
| Graph residual and spatial reconstruction | 1024 × 16 × 16 |
| Four transposed convolutions | 512×32² → 256×64² → 128×128² → 64×256² |
| Encoder skip + final convolution | 3 × 256 × 256 |

The four-convolution encoder residual block and dropout 0.4 follow the author code; the PDF does not spell them out. The block brings the model to approximately the reported **31.18 million parameters**. The final skip is added **before** the final RGB convolution, resolving the dimension mismatch in the PDF's pseudocode. Node-to-image reconstruction uses a transpose, correcting the public implementation's direct reshape.

Graph convolutions compute `D^(-1/2) (A + I) D^(-1/2) H W + bias` with dense batched PyTorch multiplication. This is small at 256 nodes and keeps graphs from different photographs isolated. Output values remain unconstrained for training and are clipped to [0,1] only for evaluation/display.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        layers = []
        for _ in range(2):
            layers.extend([nn.Conv2d(channels, channels, 3, padding=1), nn.LeakyReLU(0.2),
                           nn.Conv2d(channels, channels, 3, padding=1)])
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.layers(x)


class GraphConv(nn.Module):
    def __init__(self, inputs, outputs):
        super().__init__()
        self.linear = nn.Linear(inputs, outputs, bias=False)
        self.bias = nn.Parameter(torch.zeros(outputs))
        nn.init.xavier_uniform_(self.linear.weight)

    def forward(self, nodes, normalized_adjacency):
        return torch.bmm(normalized_adjacency, self.linear(nodes)) + self.bias


class FUGN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        channels, features = cfg["encoder_channels"], cfg["node_features"]
        self.dropout = cfg["dropout"]
        self.encoder = nn.Sequential(nn.Conv2d(3, channels, 3, padding=1), nn.LeakyReLU(0.1),
                                     nn.Conv2d(channels, channels, 3, padding=1), ResidualBlock(channels))
        self.down = nn.Conv2d(channels, features, cfg["block_size"], stride=cfg["block_size"])
        self.gcn1 = GraphConv(features, cfg["gcn_hidden"])
        self.gcn2 = GraphConv(cfg["gcn_hidden"], features)
        layers = []
        widths = [features, features // 2, features // 4, features // 8, channels]
        for before, after in zip(widths, widths[1:]):
            layers.extend([nn.ConvTranspose2d(before, after, 4, stride=2, padding=1),
                           nn.LeakyReLU(0.1)])
        self.decoder = nn.Sequential(*layers)
        self.final = nn.Conv2d(channels, 3, 3, padding=1)

    def forward(self, images, adjacency):
        encoded = self.encoder(images)
        down = self.down(encoded)
        nodes = features_to_nodes(down)
        if adjacency.shape != (images.shape[0], nodes.shape[1], nodes.shape[1]):
            raise ValueError("Graph batch/node dimensions do not match encoder features")
        norm = normalize_graph(adjacency).to(nodes.dtype)
        hidden = F.relu(self.gcn1(nodes, norm))
        hidden = F.dropout(hidden, p=self.dropout, training=self.training)
        nodes = self.gcn2(hidden, norm)
        graph_features = nodes_to_features(nodes, down.shape[-2], down.shape[-1]) + down
        return self.final(self.decoder(graph_features) + encoded)


def loss_components(prediction, target, cfg):
    charbonnier = ((prediction - target).square() + cfg["charbonnier_epsilon"] ** 2).sqrt().mean()
    # SSIM must remain differentiable and see the unclipped training prediction.
    # PIQ validates [0,1], so the local implementation below intentionally does
    # not clip outputs or apply the metric-only input validation.
    size, sigma = 11, 1.5
    axis = torch.arange(size, device=prediction.device, dtype=torch.float32) - size // 2
    gaussian = torch.exp(-axis.square() / (2 * sigma ** 2))
    gaussian /= gaussian.sum()
    window = torch.outer(gaussian, gaussian)[None, None].expand(3, 1, size, size).contiguous()
    p, t = prediction.float(), target.float()
    mu_p = F.conv2d(p, window, groups=3)
    mu_t = F.conv2d(t, window, groups=3)
    var_p = F.conv2d(p.square(), window, groups=3) - mu_p.square()
    var_t = F.conv2d(t.square(), window, groups=3) - mu_t.square()
    covariance = F.conv2d(p * t, window, groups=3) - mu_p * mu_t
    ssim_map = ((2 * mu_p * mu_t + 0.01 ** 2) * (2 * covariance + 0.03 ** 2)
                / ((mu_p.square() + mu_t.square() + 0.01 ** 2)
                   * (var_p + var_t + 0.03 ** 2)))
    structure = 1 - ssim_map.mean()
    total = cfg["beta"] * charbonnier + (1 - cfg["beta"]) * structure
    return total, charbonnier, structure

## 4 · Quality measures and numerical checks

Full-reference metrics use RGB tensors in [0,1]: PSNR, Gaussian-window SSIM (11×11, σ=1.5, no downsampling), and **FSIM** (`piq.fsim`, `chromatic=False`), not the unrelated pixelwise ratio called FSIM in the authors' utility file. PIQ implements phase congruency and gradient magnitude following the original FSIM implementation. [PIQ documentation](https://piq.readthedocs.io/en/latest/functions.html).

No-reference measures use the formulas from Panetta et al. (UIQM) and Yang & Sowmya (UCIQE). UIQM uses 0.0282 UICM + 0.2953 UISM + 3.5753 UIConM, 10% trimmed opponent-color means, Sobel sharpness, and 10×10 blocks. UCIQE uses 0.4680 chroma standard deviation + 0.2745 luminance contrast (99th−1st percentile) + 0.2576 mean saturation, with OpenCV floating-point CIELAB scaled by 100. Block handling, Lab scaling, and implementation choices affect absolute scores: do not directly equate these scores with the PDF's unprovided metric implementation. Higher no-reference scores do not guarantee visually better results.

PSNR for identical images is correctly +∞; constant-image no-reference metrics are finite. Only paired tests receive reference-based scores.

In [ ]:
def full_reference_metrics(prediction, reference):
    prediction, reference = prediction.float(), reference.float()
    mse = (prediction - reference).square().flatten(1).mean(1)
    psnr = -10 * torch.log10(mse)
    ssim = piq.ssim(prediction, reference, data_range=1.0, downsample=False, reduction="none")
    fsim = piq.fsim(prediction, reference, data_range=1.0, chromatic=False, reduction="none")
    return {"PSNR": psnr, "SSIM": ssim, "FSIM": fsim}


def blocks_10(array):
    # Edge-pad small/partial blocks rather than dropping image-border content.
    height, width = array.shape[:2]
    padding = ((0, (-height) % 10), (0, (-width) % 10)) + ((0, 0),) * (array.ndim - 2)
    padded = np.pad(array, padding, mode="edge")
    return [padded[y:y+10, x:x+10] for y in range(0, padded.shape[0], 10)
            for x in range(0, padded.shape[1], 10)]


def uiqm(rgb):
    rgb = np.asarray(rgb, dtype=np.float64) * 255.0
    red, green, blue = rgb[..., 0], rgb[..., 1], rgb[..., 2]
    rg, yb = red - green, (red + green) / 2 - blue
    def trimmed_stats(values):
        ordered = np.sort(values.ravel())
        trim = int(0.1 * len(ordered))
        kept = ordered[trim:len(ordered)-trim] if trim else ordered
        mean = kept.mean()
        return mean, ((ordered - mean) ** 2).mean()
    mean_rg, var_rg = trimmed_stats(rg)
    mean_yb, var_yb = trimmed_stats(yb)
    uicm = -0.0268 * np.hypot(mean_rg, mean_yb) + 0.1586 * np.sqrt(var_rg + var_yb)
    def sharpness(channel):
        gx = cv2.Sobel(channel, cv2.CV_64F, 1, 0, ksize=3)
        gy = cv2.Sobel(channel, cv2.CV_64F, 0, 1, ksize=3)
        magnitude = np.hypot(gx, gy)
        maximum = magnitude.max()
        if maximum <= 1e-12:
            return 0.0
        edge_weighted = channel * (magnitude / maximum * 255.0)
        contrasts = []
        for block in blocks_10(edge_weighted):
            low, high = block.min(), block.max()
            # Zero-valued edge pixels make EME undefined; use the conventional
            # unit floor in this 0..65025 edge-weighted intensity convention.
            contrasts.append(2 * np.log(max(high, 1.0) / max(low, 1.0)))
        return float(np.mean(contrasts))
    uism = sum(weight * sharpness(channel) for weight, channel in
               zip((0.299, 0.587, 0.114), (red, green, blue)))
    contrast_terms = []
    for block in blocks_10(rgb):
        low, high = block.min(), block.max()
        ratio = (high - low) / (high + low) if high + low > 1e-12 else 0.0
        contrast_terms.append(-ratio * np.log(ratio) if ratio > 1e-12 else 0.0)
    uiconm = float(np.mean(contrast_terms))
    return float(0.0282 * uicm + 0.2953 * uism + 3.5753 * uiconm)


def uciqe(rgb):
    lab = cv2.cvtColor(np.asarray(rgb, dtype=np.float32), cv2.COLOR_RGB2LAB)
    luminance, a, b = lab[..., 0] / 100.0, lab[..., 1] / 100.0, lab[..., 2] / 100.0
    chroma = np.hypot(a, b)
    denominator = np.sqrt(chroma ** 2 + luminance ** 2)
    saturation = np.divide(chroma, denominator, out=np.zeros_like(chroma), where=denominator > 1e-12)
    contrast = np.quantile(luminance, 0.99) - np.quantile(luminance, 0.01)
    return float(0.4680 * chroma.std() + 0.2745 * contrast + 0.2576 * saturation.mean())


def no_reference_metrics(rgb):
    return {"UIQM": uiqm(rgb), "UCIQE": uciqe(rgb)}


def run_self_checks(device="cpu", full_model=False):
    # One runnable check covers the failure-prone cross-component contracts.
    import tempfile
    c = make_config()
    c["num_workers"] = 0
    small = dict(c, encoder_channels=4, node_features=32, gcn_hidden=48, dropout=0.0)
    random_images = torch.rand(2, 3, 32, 32, device=device)
    local = dict(small, neighbors=2, hop=3)
    graph = build_graph(random_images, local)
    assert torch.equal(graph.diagonal(dim1=-2, dim2=-1), torch.zeros(2, 4, device=device))
    assert torch.all(graph.sum(-1) == 2)
    assert torch.equal(graph[:1], build_graph(random_images[:1], local))
    constant_graph = build_graph(torch.zeros_like(random_images), local)
    assert torch.isfinite(normalize_graph(constant_graph)).all()
    assert torch.all(constant_graph.sum(-1) == 2)
    numbered = torch.arange(2 * 7 * 3 * 5).reshape(2, 7, 3, 5)
    assert torch.equal(nodes_to_features(features_to_nodes(numbered), 3, 5), numbered)
    tiny = FUGN(small).to(device).eval()
    prediction = tiny(random_images, graph)
    assert prediction.shape == random_images.shape
    assert torch.allclose(prediction[:1], tiny(random_images[:1], graph[:1]), atol=1e-5)
    # Both members of a synthetic identical pair must get exactly the same crop.
    with tempfile.TemporaryDirectory() as temporary:
        path = Path(temporary) / "paired.png"
        array = np.random.default_rng(42).integers(0, 256, size=(290, 320, 3), dtype=np.uint8)
        Image.fromarray(array).save(path)
        dataset = PairedImages([(path, path, "paired")], c, training=True)
        left, right, _ = dataset[0]
        assert torch.equal(left, right) and left.shape == (3, 256, 256)
        optimizer = torch.optim.Adam(tiny.parameters(), lr=1e-3)
        before = tiny.final.weight.detach().clone()
        loss, _, _ = loss_components(prediction, random_images, c)
        assert torch.isfinite(loss)
        loss.backward()
        assert all(torch.isfinite(p.grad).all() for p in tiny.parameters() if p.grad is not None)
        optimizer.step()
        assert not torch.equal(before, tiny.final.weight)
        checkpoint_path = Path(temporary) / "checkpoint.pth"
        torch.save({"model": tiny.state_dict(), "optimizer": optimizer.state_dict()}, checkpoint_path)
        restored = FUGN(small).to(device).eval()
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
        restored.load_state_dict(checkpoint["model"])
        restored_optimizer = torch.optim.Adam(restored.parameters())
        restored_optimizer.load_state_dict(checkpoint["optimizer"])
        assert torch.equal(tiny.final.weight, restored.final.weight)
        assert torch.allclose(tiny(random_images, graph), restored(random_images, graph))
    example = torch.rand(1, 3, 64, 64, device=device)
    identity = full_reference_metrics(example, example)
    assert torch.isinf(identity["PSNR"]).all()
    assert abs(identity["SSIM"].item() - 1) < 1e-5
    assert abs(identity["FSIM"].item() - 1) < 1e-5
    # Check SSIM loss against the independently implemented library metric.
    _, _, ssim_loss = loss_components(example * 0.9, example, c)
    ssim_reference = piq.ssim(example * 0.9, example, downsample=False)
    assert abs(ssim_loss.item() - (1 - ssim_reference.item())) < 2e-5
    for value in (0.0, 0.5, 1.0):
        assert all(np.isfinite(v) for v in no_reference_metrics(np.full((32, 32, 3), value)).values())
    black, white = torch.zeros_like(example), torch.ones_like(example)
    assert abs((-10 * torch.log10((black - white).square().mean())).item()) < 1e-7
    if full_model:
        model = FUGN(c).to(device).eval()
        params = sum(p.numel() for p in model.parameters())
        assert params == 31_183_007, params
        with torch.inference_mode():
            image = torch.rand(1, 3, 256, 256, device=device)
            result = model(image, build_graph(image, c))
        assert result.shape == image.shape and torch.isfinite(result).all()
        del model
    print("Self-checks passed: graph, batch isolation, crop, reshape, gradients, checkpoint and metrics.")

In [ ]:
seed_everything(cfg["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if os.environ.get("FUGN_REQUIRE_CUDA") == "1" and device.type != "cuda":
    raise RuntimeError("This remote experiment requires a CUDA GPU")
if device.type == "cpu":
    print("CPU execution: the full training run can be very slow.")
run_self_checks(str(device), full_model=True)
runtime = {"python": platform.python_version(), "torch": torch.__version__,
           "cuda": torch.version.cuda, "device": str(device),
           "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
           "packages": {package: importlib.metadata.version(package)
                        for package in ["torch", "torchvision", "numpy", "piq", "opencv-python-headless", "Pillow"]}}
(output_dir / "runtime.json").write_text(json.dumps(runtime, indent=2))
print(runtime)

## 5 · Supervised optimization — Equations 10–13

The loss is **0.5 mean(Charbonnier) + 0.5 (1 − mean SSIM)** with ε=1e-3. Means make the relative weights independent of image and batch size. Adam uses its standard β=(0.9,0.999), ε=1e-8, with no weight decay or scheduler. Gradient clipping at norm 0.1 follows the author code.

The pilot runs ten epochs with learning rate 2e-4. Automatic mixed precision is enabled on CUDA; the SSIM statistics and graph construction remain float32. Validation uses unclipped outputs for loss and clipped RGB outputs for perceptual metrics. Best checkpoint selection uses validation PSNR only. The held-out test set never selects checkpoints.

Checkpoints are written atomically after every completed epoch and include model/optimizer/scaler state, random-generator states, split fingerprint and configuration. Resuming repeats neither completed epochs nor their random crop sequence. Changing the run's model, loss, optimizer, precision or split settings is rejected; use a new output directory instead.

In [ ]:
def atomic_checkpoint(payload, path):
    temporary = path.with_suffix(".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)


def compatibility_settings(cfg):
    ignored = {"epochs", "resume", "output_dir", "data_dir", "commit", "num_workers"}
    return {key: value for key, value in cfg.items() if key not in ignored}


def fit_model(cfg, splits, device, output_dir):
    seed_everything(cfg["seed"])
    model = FUGN(cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["learning_rate"])
    amp_enabled = cfg["amp"] and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    history, first_epoch, best_psnr = [], 1, -math.inf
    latest, best = output_dir / "latest.pth", output_dir / "best.pth"
    if latest.exists():
        if not cfg["resume"]:
            raise FileExistsError("Output already has a checkpoint; use a new output directory or enable resume")
        checkpoint = torch.load(latest, map_location=device, weights_only=True)
        if checkpoint["split_fingerprint"] != splits["manifest"]["fingerprint"]:
            raise ValueError("Checkpoint dataset split differs")
        if compatibility_settings(checkpoint["config"]) != compatibility_settings(cfg):
            raise ValueError("Checkpoint training settings differ; use a new output directory")
        if checkpoint["epoch"] > cfg["epochs"]:
            raise ValueError("Requested epochs precede the existing checkpoint")
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scaler.load_state_dict(checkpoint["scaler"])
        history, best_psnr = checkpoint["history"], checkpoint["best_psnr"]
        first_epoch = checkpoint["epoch"] + 1
        torch.set_rng_state(checkpoint["torch_rng"].cpu())
        if device.type == "cuda":
            torch.cuda.set_rng_state_all([state.cpu() for state in checkpoint["cuda_rng"]])
        random.setstate(checkpoint["python_rng"])
        print(f"Resuming from completed epoch {checkpoint['epoch']}", flush=True)
    train_data = PairedImages(splits["train"], cfg, training=True)
    val_data = PairedImages(splits["val"], cfg)
    val_loader = make_loader(val_data, cfg["batch_size"])
    for epoch in range(first_epoch, cfg["epochs"] + 1):
        started = time.perf_counter()
        train_data.epoch = epoch
        train_loader = make_loader(train_data, cfg["batch_size"], shuffle=True, seed=cfg["seed"] + epoch)
        model.train()
        sums = np.zeros(3, dtype=np.float64)
        seen = 0
        for step, (raw, reference, _) in enumerate(train_loader, 1):
            raw, reference = raw.to(device, non_blocking=True), reference.to(device, non_blocking=True)
            adjacency = build_graph(raw, cfg)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=amp_enabled):
                prediction = model(raw, adjacency)
            with torch.autocast(device_type=device.type, enabled=False):
                losses = loss_components(prediction.float(), reference.float(), cfg)
            if not torch.isfinite(losses[0]):
                raise FloatingPointError(f"Non-finite training loss at epoch {epoch}, step {step}")
            scaler.scale(losses[0]).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"], error_if_nonfinite=True)
            scaler.step(optimizer)
            scaler.update()
            sums += np.array([value.detach().item() for value in losses]) * raw.shape[0]
            seen += raw.shape[0]
            if step % 60 == 0 or step == len(train_loader):
                print(f"epoch {epoch}/{cfg['epochs']} step {step}/{len(train_loader)} loss={sums[0]/seen:.5f}", flush=True)
        model.eval()
        val_losses, val_psnr, val_ssim = [], [], []
        with torch.inference_mode():
            for raw, reference, _ in val_loader:
                raw, reference = raw.to(device), reference.to(device)
                prediction = model(raw, build_graph(raw, cfg))
                batch_loss = loss_components(prediction, reference, cfg)[0].item()
                val_losses.extend([batch_loss] * raw.shape[0])
                clipped = prediction.clamp(0, 1)
                mse = (clipped - reference).square().flatten(1).mean(1)
                val_psnr.extend((-10 * torch.log10(mse)).cpu().tolist())
                val_ssim.extend(piq.ssim(clipped, reference, downsample=False, reduction="none").cpu().tolist())
        row = {"epoch": epoch, "train_loss": float(sums[0] / seen), "train_charbonnier": float(sums[1] / seen),
               "train_ssim_loss": float(sums[2] / seen), "val_loss": float(np.mean(val_losses)),
               "val_psnr": float(np.mean(val_psnr)), "val_ssim": float(np.mean(val_ssim)),
               "seconds": time.perf_counter() - started, "learning_rate": optimizer.param_groups[0]["lr"]}
        if not all(np.isfinite(row[key]) for key in ("train_loss", "val_loss", "val_psnr", "val_ssim")):
            raise FloatingPointError("Non-finite epoch metrics")
        history.append(row)
        improved = row["val_psnr"] > best_psnr
        best_psnr = max(best_psnr, row["val_psnr"])
        checkpoint = {"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                      "scaler": scaler.state_dict(), "config": cfg, "history": history,
                      "split_fingerprint": splits["manifest"]["fingerprint"], "best_psnr": best_psnr,
                      "torch_rng": torch.get_rng_state(), "python_rng": random.getstate(),
                      "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else []}
        if improved:
            atomic_checkpoint(checkpoint, best)
        atomic_checkpoint(checkpoint, latest)
        pd.DataFrame(history).to_csv(output_dir / "history.csv", index=False)
        print(json.dumps(row), flush=True)
    if not best.exists():
        raise FileNotFoundError("No completed best checkpoint is available")
    checkpoint = torch.load(best, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint["model"])
    model.eval()
    return model, pd.DataFrame(history), checkpoint["epoch"]

In [ ]:
model, history, best_epoch = fit_model(cfg, splits, device, output_dir)
print(f"Best checkpoint: epoch {best_epoch}; completed {len(history)} epochs")
display(history)

## 6 · Held-out evaluation and exported photographs

Every test photograph is evaluated against its paired reference. The challenging set receives only UIQM/UCIQE. The same image tensors are used for the input baseline and FUGN. Outputs are saved as 8-bit PNGs; metric calculations use clipped floating-point predictions **before PNG quantization**. This small distinction is recorded for reproducibility.

Inference timing includes graph construction and neural inference, with CUDA synchronization, and excludes decoding, saving, plotting and quality-metric computation. The first test photograph is warmed up before timing.

In [ ]:
def evaluate_model(model, cfg, splits, device, output_dir):
    rows, examples = [], []
    (output_dir / "enhanced").mkdir(exist_ok=True)
    for group in ("test", "challenging"):
        (output_dir / "enhanced" / group).mkdir(exist_ok=True)
        dataset = PairedImages(splits[group], cfg)
        for index in range(len(dataset)):
            raw, reference, name = dataset[index]
            raw_batch = raw[None].to(device)
            with torch.inference_mode():
                if index == 0:
                    model(raw_batch, build_graph(raw_batch, cfg))
                if device.type == "cuda":
                    torch.cuda.synchronize()
                start = time.perf_counter()
                prediction = model(raw_batch, build_graph(raw_batch, cfg)).float().clamp(0, 1)
                if device.type == "cuda":
                    torch.cuda.synchronize()
                milliseconds = (time.perf_counter() - start) * 1000
                baseline_metrics, output_metrics = {}, {}
                if reference.numel():
                    ref_batch = reference[None].to(device)
                    baseline_metrics = {key: values.item() for key, values in full_reference_metrics(raw_batch, ref_batch).items()}
                    output_metrics = {key: values.item() for key, values in full_reference_metrics(prediction, ref_batch).items()}
            input_rgb = raw.permute(1, 2, 0).numpy()
            output_rgb = prediction[0].cpu().permute(1, 2, 0).numpy()
            reference_rgb = reference.permute(1, 2, 0).numpy() if reference.numel() else None
            baseline_metrics.update(no_reference_metrics(input_rgb))
            output_metrics.update(no_reference_metrics(output_rgb))
            if not all(np.isfinite(value) for value in [*baseline_metrics.values(), *output_metrics.values()]):
                raise FloatingPointError(f"Non-finite evaluation metric: {group}/{name}")
            for method, metrics in [("Input", baseline_metrics), ("FUGN", output_metrics)]:
                rows.append({"split": group, "image": name, "method": method,
                             **metrics, "inference_ms": milliseconds if method == "FUGN" else np.nan})
            Image.fromarray(np.rint(output_rgb * 255).astype(np.uint8)).save(output_dir / "enhanced" / group / f"{name}.png")
            examples.append({"split": group, "name": name, "input": input_rgb,
                             "output": output_rgb, "reference": reference_rgb,
                             "input_metrics": baseline_metrics, "output_metrics": output_metrics})
            if (index + 1) % 15 == 0:
                print(f"Evaluated {group}: {index+1}/{len(dataset)}", flush=True)
    metrics = pd.DataFrame(rows)
    metrics.to_csv(output_dir / "metrics.csv", index=False)
    summary = metrics.groupby(["split", "method"])[["PSNR", "SSIM", "FSIM", "UIQM", "UCIQE"]].agg(["mean", "std"])
    summary.to_csv(output_dir / "metric_summary.csv")
    return metrics, examples, summary

In [ ]:
metrics, examples, metric_summary = evaluate_model(model, cfg, splits, device, output_dir)
display(metric_summary.round(4))

## 7 · Graphs, comparisons, and offline HTML report

The report contains all 90 held-out triplets and all 60 challenging pairs, including weak or degraded outputs. Test examples are presented in fixed filename order, never selected for high scores. Summary plots compare input and output, show per-image score changes, and include representative histograms and graph connectivity.

**Interpretation:** this is a single seeded pilot, not the paper's ten-run result. A higher average score does not imply every image improves. The HTML records the best validation epoch, observed changes, and the number of test images whose PSNR improves. No claim of reproducing the PDF's PSNR 26.52 dB is made.

In [ ]:
def figure_image(fig, path, alt):
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f'<img alt="{html.escape(alt, quote=True)}" src="data:image/png;base64,{encoded}" loading="lazy">'


def build_report(cfg, runtime, history, best_epoch, metrics, examples, summary, model, device, output_dir):
    figures = output_dir / "figures"
    comparisons = output_dir / "comparisons"
    figures.mkdir(exist_ok=True)
    comparisons.mkdir(exist_ok=True)
    sections = []
    def add_figure(fig, filename, alt):
        sections.append(figure_image(fig, figures / filename, alt))
        display(fig)
        plt.close(fig)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history.epoch, history.train_loss, label="Training")
    axes[0].plot(history.epoch, history.val_loss, label="Validation")
    axes[0].set(title="Optimization loss", xlabel="Epoch", ylabel="Combined loss")
    axes[0].legend()
    axes[1].plot(history.epoch, history.val_psnr, color="#077f8c")
    axes[1].axvline(best_epoch, color="gray", linestyle="--", label="Best checkpoint")
    axes[1].set(title="Validation PSNR", xlabel="Epoch", ylabel="dB")
    axes[1].legend()
    axes[2].plot(history.epoch, history.val_ssim, color="#b35a15")
    axes[2].set(title="Validation SSIM", xlabel="Epoch", ylabel="SSIM")
    fig.tight_layout()
    add_figure(fig, "training.png", "Training and validation loss, PSNR and SSIM by epoch")
    paired = metrics[metrics.split == "test"]
    means = paired.groupby("method")[["PSNR", "SSIM", "FSIM", "UIQM", "UCIQE"]].mean()
    fig, axes = plt.subplots(1, 5, figsize=(16, 4))
    for axis, metric in zip(axes, means.columns):
        axis.bar(["Input", "FUGN"], means.loc[["Input", "FUGN"], metric], color=["#8d9b9f", "#077f8c"])
        axis.set(title=metric, ylabel="dB" if metric == "PSNR" else "Score")
    fig.suptitle("Held-out paired tests · means over 90 photographs")
    fig.tight_layout()
    add_figure(fig, "metrics.png", "Mean input and FUGN quality scores on 90 held-out paired images")
    pivots = {metric: paired.pivot(index="image", columns="method", values=metric)
              for metric in ("PSNR", "SSIM", "FSIM")}
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for axis, (metric, values) in zip(axes, pivots.items()):
        delta = values["FUGN"] - values["Input"]
        axis.hist(delta, bins=18, color="#077f8c", alpha=0.85)
        axis.axvline(0, color="#b35a15", linestyle="--")
        axis.set(title=f"{metric} · {int((delta > 0).sum())}/90 improved",
                 xlabel="FUGN − input", ylabel="Photographs")
    fig.tight_layout()
    add_figure(fig, "improvement_distribution.png", "Distribution of per-image PSNR, SSIM and FSIM changes")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for axis, metric in zip(axes, ("UIQM", "UCIQE")):
        challenge_means = metrics[metrics.split == "challenging"].groupby("method")[metric].mean()
        axis.bar(["Input", "FUGN"], challenge_means.loc[["Input", "FUGN"]], color=["#8d9b9f", "#077f8c"])
        axis.set(title=f"Challenging-60 · {metric}", ylabel="Score")
    fig.tight_layout()
    add_figure(fig, "challenging_metrics.png", "No-reference scores on the 60 challenging images")
    # Fixed representative examples span the ordered test list, without ranking outputs.
    selected = [examples[i] for i in (0, 29, 59, 89)]
    fig, axes = plt.subplots(4, 3, figsize=(14, 12))
    for row, example in enumerate(selected):
        for channel, color in enumerate(("red", "green", "blue")):
            axis = axes[row, channel]
            for key, linestyle, label in [("input", ":", "Input"), ("output", "-", "FUGN"), ("reference", "--", "Reference")]:
                counts, edges = np.histogram(example[key][..., channel], bins=64, range=(0, 1), density=True)
                axis.plot((edges[:-1] + edges[1:]) / 2, counts, color=color, linestyle=linestyle, label=label)
            axis.set(title=f"{example['name']} · {color}", xlabel="Channel intensity", ylabel="Density")
            if row == 0:
                axis.legend()
    fig.tight_layout()
    add_figure(fig, "rgb_histograms.png", "RGB intensity histograms for four fixed held-out photographs")
    image = torch.from_numpy(examples[0]["input"]).permute(2, 0, 1)[None].to(device)
    adjacency, similarity = build_graph(image, cfg, return_similarity=True)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(examples[0]["input"])
    for position in range(16, 256, 16):
        axes[0].axhline(position - 0.5, color="white", alpha=0.4, linewidth=0.4)
        axes[0].axvline(position - 0.5, color="white", alpha=0.4, linewidth=0.4)
    axes[0].set_title("16×16 patch partition")
    axes[0].axis("off")
    axes[1].imshow(similarity[0].cpu(), cmap="viridis", vmin=0, vmax=1)
    axes[1].set(title="Sobel + Gaussian similarity", xlabel="Node", ylabel="Node")
    axes[2].imshow(adjacency[0].cpu(), cmap="gray_r", interpolation="nearest")
    axes[2].set(title="Five outgoing neighbors / node", xlabel="Node", ylabel="Node")
    fig.tight_layout()
    add_figure(fig, "graph.png", "Patch partition, operator similarity and resulting graph adjacency")
    psnr_delta = pivots["PSNR"]["FUGN"] - pivots["PSNR"]["Input"]
    report_summary = {
        "status": "completed", "preset": cfg["preset"], "epochs_completed": int(history.epoch.max()),
        "best_validation_epoch": int(best_epoch), "learning_rate": cfg["learning_rate"],
        "parameters": sum(p.numel() for p in model.parameters()),
        "paired_comparisons": sum(e["split"] == "test" for e in examples),
        "challenging_comparisons": sum(e["split"] == "challenging" for e in examples),
        "paired_psnr_input": float(means.loc["Input", "PSNR"]),
        "paired_psnr_fugn": float(means.loc["FUGN", "PSNR"]),
        "paired_ssim_input": float(means.loc["Input", "SSIM"]),
        "paired_ssim_fugn": float(means.loc["FUGN", "SSIM"]),
        "psnr_improved_images": int((psnr_delta > 0).sum()),
        "training_minutes": float(history.seconds.sum() / 60),
        "mean_inference_ms": float(metrics.loc[metrics.method == "FUGN", "inference_ms"].mean()),
        "source_commit": cfg["commit"], "report": "fugn_uieb_pilot.html",
    }
    (output_dir / "summary.json").write_text(json.dumps(report_summary, indent=2))
    gallery_sections = []
    for group, columns in [("test", 3), ("challenging", 2)]:
        group_examples = [example for example in examples if example["split"] == group]
        gallery_sections.append(f'<h2 id="{group}">{"Paired tests · 90 triplets" if group == "test" else "Challenging set · 60 pairs"}</h2>')
        for start in range(0, len(group_examples), 5):
            page = group_examples[start:start+5]
            fig, axes = plt.subplots(len(page), columns, figsize=(4 * columns, 3.7 * len(page)), squeeze=False)
            for row, example in enumerate(page):
                entries = [("input", "Input"), ("output", "FUGN")]
                if group == "test":
                    entries.append(("reference", "Reference"))
                for col, (key, title) in enumerate(entries):
                    axes[row, col].imshow(example[key])
                    label = f"{example['name']} · {title}"
                    if key in {"input", "output"} and group == "test":
                        values = example[f"{key}_metrics"]
                        label += f"\nPSNR {values['PSNR']:.2f} dB · SSIM {values['SSIM']:.3f}"
                    axes[row, col].set_title(label, fontsize=10)
                    axes[row, col].axis("off")
            fig.tight_layout()
            alt = f"{group} comparisons {start+1}–{start+len(page)}: " + ", ".join(e["name"] for e in page)
            gallery_sections.append(figure_image(fig, comparisons / f"{group}_{start//5+1:02d}.png", alt))
            display(fig)
            plt.close(fig)
    summary_table = summary.round(4).to_html(classes="metrics", border=0, na_rep="—")
    data_table = metrics.round(5).to_html(index=False, classes="metrics", border=0, na_rep="—")
    if cfg["preset"] == "pilot":
        scope_note = f"Single seeded {len(history)}-epoch pilot at learning rate {cfg['learning_rate']:g}; not the paper's ten-run benchmark."
    else:
        scope_note = f"Single seeded {len(history)}-epoch run using the PDF learning rate; unpublished graph settings and the exact author split remain unrecoverable."
    report = """<!doctype html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>FUGN · UIEB results</title><style>
:root{color-scheme:light;font-family:system-ui,-apple-system,sans-serif;color:#163138;background:#f4f7f7}
body{max-width:1220px;margin:auto;padding:32px 24px;line-height:1.6}h1{font-size:2.4rem;line-height:1.15}
h2{margin-top:48px;border-bottom:1px solid #ccd8da;padding-bottom:10px}a{color:#086b79}
nav{display:flex;gap:20px;flex-wrap:wrap;margin:24px 0}img{display:block;width:100%;height:auto;background:white;margin:24px 0;border:1px solid #dce4e5}
.note{padding:16px 20px;border-left:4px solid #087f8c;background:#e7f0f1}.table-wrap{overflow:auto;background:white;padding:12px}
table{border-collapse:collapse;font-size:.87rem;width:100%}th,td{padding:8px 10px;border-bottom:1px solid #dde5e7;text-align:right;white-space:nowrap}
th{text-align:center}pre{white-space:pre-wrap;overflow-wrap:anywhere;background:#e8efef;padding:18px}details{margin:24px 0}
@media print{body{max-width:none;padding:0}nav{display:none}img{break-inside:avoid}}
</style></head><body>"""
    report += '<p>REPRODUCIBLE EXPERIMENT · REAL UNDERWATER PHOTOGRAPHS</p><h1>FUGN · UIEB enhancement results</h1>'
    report += f'<p class="note">{html.escape(scope_note)} All 150 test comparisons are included, without filtering for improvements.</p>'
    report += '<nav><a href="#overview">Results</a><a href="#method">Method</a><a href="#graphs">Graphs</a><a href="#test">90 paired tests</a><a href="#challenging">60 challenging images</a><a href="#records">Per-image metrics</a></nav>'
    report += '<h2 id="overview">Measured results</h2>'
    report += f'<p>Best validation checkpoint: epoch <strong>{best_epoch}</strong>. Held-out PSNR: <strong>{means.loc["Input", "PSNR"]:.2f} → {means.loc["FUGN", "PSNR"]:.2f} dB</strong>; SSIM: <strong>{means.loc["Input", "SSIM"]:.4f} → {means.loc["FUGN", "SSIM"]:.4f}</strong>. PSNR improves on {int((psnr_delta > 0).sum())} of 90 paired tests. Training took {history.seconds.sum()/60:.1f} minutes on {html.escape(str(runtime["gpu"] or runtime["device"]))}.</p>'
    report += '<p>Values below are means and sample standard deviations across photographs, not variation across training runs. Reference-based scores are unavailable for the challenging set.</p><div class="table-wrap">' + summary_table + '</div>'
    report += '<h2 id="method">Method and reproducibility</h2><p>The complete 31,183,007-parameter model combines a 64-channel CNN encoder, a 256-node graph with 1024-dimensional features, two GCN layers through 1500 hidden features, four transposed-convolution decoder stages, and residual fusion. Graphs use Sobel and Gaussian RBF similarities, five outgoing non-self neighbors, full-image candidate search, and self-loops during degree normalization. Gaussian σ=1 (7×7); per-image median-distance RBF bandwidths are explicit defaults omitted from the PDF.</p>'
    report += '<p>Training uses Adam, batch size 2, equally weighted mean Charbonnier and SSIM losses, dropout 0.4, gradient clipping 0.1, and CUDA mixed precision. The encoder residual block follows the author code. The PDF training preset uses 1e-6 for 100 epochs; this report records the settings actually used below.</p>'
    report += '<p>UIEB is split into 720 training, 80 validation, and 90 paired test photographs (seed 42), plus 60 reference-free challenging photographs. Training uses aligned random crops; evaluation resizes both members to 256×256. Metrics use clipped floats before PNG quantization. FSIM is phase-congruency based, with chromatic=False. UIQM uses 10×10 edge-padded blocks and unit floors for edge-weighted EME. UCIQE uses float Lab channels divided by 100. These conventions and the unpublished author split limit direct comparison with the paper. No-reference metrics can reward artifacts; inspect the photographs.</p>'
    report += '<p>Sources: <a href="https://doi.org/10.1016/j.inffus.2024.102857">FUGN paper</a> · <a href="https://github.com/whZ62/FUGN">Author code</a> · <a href="https://li-chongyi.github.io/proj_benchmark.html">UIEB authors</a> · <a href="https://piq.readthedocs.io/en/latest/functions.html">PIQ metrics</a>. Dataset images are for private academic use and are excluded from the public source repository.</p>'
    report += '<details><summary>Exact run configuration and software versions</summary><pre>' + html.escape(json.dumps({"config": cfg, "runtime": runtime, "summary": report_summary}, indent=2)) + '</pre></details>'
    report += '<h2 id="graphs">Training, quality and graph diagnostics</h2>' + ''.join(sections)
    report += ''.join(gallery_sections)
    report += '<h2 id="records">Every measured image</h2><div class="table-wrap">' + data_table + '</div>'
    report += '<p>Every plot and photograph is embedded in this file. No network connection, server, JavaScript library, or external asset is needed to view the report.</p></body></html>'
    report_path = output_dir / "fugn_uieb_pilot.html"
    report_path.write_text(report, encoding="utf-8")
    assert report_summary["paired_comparisons"] == 90
    assert report_summary["challenging_comparisons"] == 60
    assert len(list(comparisons.glob("test_*.png"))) == 18
    assert len(list(comparisons.glob("challenging_*.png"))) == 12
    assert report.count('src="data:image/png;base64,') == 36
    assert '<script' not in report and 'src="http' not in report
    print(f"Saved offline report: {report_path} ({report_path.stat().st_size / 1024**2:.1f} MiB)")
    return report_path, report_summary

In [ ]:
report_path, experiment_summary = build_report(
    cfg, runtime, history, best_epoch, metrics, examples, metric_summary, model, device, output_dir)
display(Markdown(f"**Finished.** Open `{report_path}` for the complete offline report."))
print(json.dumps(experiment_summary, indent=2))

## References and artifact guide

- Xu et al. (2025), *Fusion-based graph neural networks for synergistic underwater image enhancement*. [DOI: 10.1016/j.inffus.2024.102857](https://doi.org/10.1016/j.inffus.2024.102857).
- Li et al. (2020), *An Underwater Image Enhancement Benchmark Dataset and Beyond*. IEEE TIP 29, 4376–4389. [Dataset](https://li-chongyi.github.io/proj_benchmark.html).
- Zhang et al. (2011), *FSIM: A Feature Similarity Index for Image Quality Assessment*. IEEE TIP 20(8), 2378–2386. [DOI](https://doi.org/10.1109/TIP.2011.2109730).
- Panetta, Gao & Agaian (2016), *Human-Visual-System-Inspired Underwater Image Quality Measures*. IEEE JOE 41(3), 541–551. [DOI](https://doi.org/10.1109/JOE.2015.2469915).
- Yang & Sowmya (2015), *An Underwater Color Image Quality Evaluation Metric*. IEEE TIP 24(12), 6062–6071. [DOI](https://doi.org/10.1109/TIP.2015.2491020).

The output directory contains the standalone HTML report, `history.csv`, `metrics.csv`, `metric_summary.csv`, split/configuration/runtime JSON files, `best.pth`, `latest.pth`, 150 enhanced PNGs, 30 comparison sheets, and six diagnostic figures. Modal also saves `FUGN_UIEB_executed.ipynb` after every completed cell, including if a later cell fails. An extended run can resume from an epoch-boundary checkpoint with identical settings and a larger epoch limit.